# Confiança de evidência — análise interna

Este notebook visualiza apenas o split interno de seleção. Ele recusa qualquer artefato marcado como uso do teste final e não deve ser citado como resultado final.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd() if (Path.cwd() / 'results').is_dir() else Path.cwd().parent
CONFIDENCE = ROOT / 'results/runs/20260721-021033_confidence-logreg_27fc4ef8'
ROBUSTNESS = ROOT / 'results/runs/20260721-021427_robustness-k5_1dacdb1b'
FIGURES = ROOT / 'figures/internal_selection_v1'


In [ ]:
confidence_manifest = json.loads((CONFIDENCE / 'manifest.json').read_text())
robustness_manifest = json.loads((ROBUSTNESS / 'manifest.json').read_text())
figure_manifest = json.loads((FIGURES / 'manifest.json').read_text())
assert confidence_manifest['final_test_used'] is False
assert robustness_manifest['final_test_used'] is False
assert figure_manifest['final_test_used'] is False
print('Escopo confirmado: seleção interna; teste final não utilizado.')


In [ ]:
metrics = json.loads((CONFIDENCE / 'metrics.json').read_text())['k5']
rows = []
for method, values in metrics['calibration_methods'].items():
    classification = values['classification_at_0_5']
    rows.append({
        'método': method,
        'ROC-AUC': classification['roc_auc'],
        'PR-AUC': classification['pr_auc'],
        'Brier': values['brier'],
        'ECE-10': values['ece'],
        'AURC': values['selective']['aurc_discrete'],
    })
pd.DataFrame(rows).round(3)


In [ ]:
for name in ['reliability_k5.png', 'risk_coverage_k5.png', 'feature_ablation_k5.png']:
    display(Image(filename=FIGURES / name, width=760))


In [ ]:
errors = pd.read_csv(FIGURES / 'error_analysis_k5.csv')
pd.crosstab(errors['scenario_type'], errors['error_type'], margins=True)


## Leitura provisória

Os resultados orientam o desenho dos negativos difíceis e o congelamento do protocolo. A conclusão científica depende da aplicação única da configuração congelada ao holdout final.